# Day 5 - Code breaks, block 1: recurrent networks

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the number marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

**Data:** *ECG5000* - 5 000 heartbeats from one long ECG recording, each cut to 140 time steps (the same data as in
the lab this afternoon).

## Setup - run these two cells first
The first cell loads PyTorch and the heartbeats (about 10 MB, downloaded once).

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn


def load_ecg5000(root="data/ecg5000"):
    """All 5000 beats: X (5000, 140, 1) = (beats, time steps, features), y (5000,) = class 0...4."""
    root = Path(root)
    if not (root / "ECG5000_TEST.txt").exists():
        url = "https://www.timeseriesclassification.com/aeon-toolkit/ECG5000.zip"
        zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(root)
    rows = [[float(v) for v in line.split()] for name in ("ECG5000_TRAIN.txt", "ECG5000_TEST.txt")
            for line in (root / name).read_text().splitlines() if line.strip()]
    data = torch.tensor(rows)
    return data[:, 1:].unsqueeze(-1), data[:, 0].long() - 1


X, y = load_ecg5000()
print("ready - PyTorch", torch.__version__, "-", len(X), "heartbeats of", X.shape[1], "time steps")

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `show_memory(beat, out)` - draws one heartbeat and, below it, what the RNN memory does at every time step
- `make_rnn_cell(size)` - a small RNN cell whose memory weights are scaled to `size` (0.5 = small, 3 = large)
- `plot_gradients(sizes)` - draws the size of the gradient at every time step (log scale)
- `show_clipping(before, after)` - draws some numbers of the gradient before and after clipping

In [ ]:
def show_memory(beat, out):
    fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
    axes[0].plot(beat[:, 0], color="#23373B")
    axes[0].set_ylabel("ECG input")
    axes[1].plot(out.detach())
    axes[1].set_ylabel("memory h_t\n(one line per unit)")
    axes[1].set_xlabel("time step t")
    plt.tight_layout()
    plt.show()


def make_rnn_cell(size, hidden=32):
    torch.manual_seed(0)
    cell = nn.RNNCell(1, hidden)
    with torch.no_grad():
        cell.weight_hh *= size / torch.linalg.eigvals(cell.weight_hh).abs().max()
    return cell


def plot_gradients(sizes):
    plt.figure(figsize=(7, 3.2))
    plt.semilogy(range(1, len(sizes) + 1), sizes, color="#EB811B", lw=2)
    plt.xlabel("time step t  (the loss is computed at the last step)")
    plt.ylabel("size of the gradient at h_t")
    plt.show()


def show_clipping(before, after):
    fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
    for ax, g, title, col in [(axes[0], before, "before clipping", "#C0392B"), (axes[1], after, "after clipping", "#2E86AB")]:
        ax.bar(range(len(g)), g, color=col)
        ax.axhline(0, color="#888", lw=0.8)
        ax.set_title(f"{title}  (largest: {g.abs().max():.3g})")
        ax.set_xlabel("16 numbers of the gradient")
    plt.tight_layout()
    plt.show()


print("helpers ready")

## 💻 Code break 1 - An RNN with one number of memory
The RNN rule from the slide, for **one** memory number `h`:  `h = tanh(w_h · h + w_x · x + b)`.
The input is 1 at the first time step, then only zeros. How long does the RNN remember the 1?

In [ ]:
INPUTS = [1, 0, 0, 0, 0, 0, 0, 0]    # 👈 the sequence x_1, x_2, ...
W_H = 0.5                            # 👈 weight of the old memory
W_X = 1.0                            # weight of the new input
B = 0.0                              # bias

h = torch.tensor(0.0)                # the memory starts empty: h_0 = 0
for t, x in enumerate(INPUTS, start=1):
    h = torch.tanh(W_H * h + W_X * x + B)
    print(f"t = {t}   input x = {x}   memory h = {h:.3f}")

🔧 **Change it:**
1. `W_H = 0.9` - how much is left of the 1 after 8 steps?
2. `INPUTS = [1, 0, 0, 1, 0, 0, 0, 0]` - what happens at t = 4?
3. `W_H = 2.0` - does the memory still fade?

<details><summary>👀 What you should see</summary>

- W_H = 0.5: **0.762, 0.363, 0.180, 0.090, 0.045, 0.022, 0.011, 0.006** - the memory halves at every step;
  after 8 steps the 1 is almost forgotten
- W_H = 0.9: **0.762, 0.595, 0.490, 0.414, 0.356, 0.310, 0.272, 0.240** - it fades more slowly
- a second 1 at t = 4: the memory jumps up again (0.797 with W_H = 0.5): new input is added to the old memory
- W_H = 2.0: **0.762, 0.909, 0.949, 0.956, ...** - the memory stays high and never goes back to 0. Now it cannot
  forget anything any more - this is also bad (and the gradients explode, code break 3)
- the RNN uses the **same** W_H and W_X at every time step - only `h` changes
</details>

## 💻 Code break 2 - `nn.RNN` on real heartbeats: shapes first
PyTorch wants a sequence as **(batch, time steps, features)**. One ECG beat: 140 time steps, 1 number per step.

In [ ]:
HIDDEN = 16          # 👈 size of the memory h
STEPS = 140          # 👈 how many time steps of every beat we use

beats = X[:8, :STEPS]                                       # 8 heartbeats
rnn = nn.RNN(input_size=1, hidden_size=HIDDEN, batch_first=True)
out, h_last = rnn(beats)                                    # out = the memory at every time step, h_last = the memory at the last time step
scores = nn.Linear(HIDDEN, 5)(out[:, -1])                   # many-to-one: last memory -> 5 class scores
print("input: ", tuple(beats.shape), "  out:", tuple(out.shape), "  scores:", tuple(scores.shape))
print("parameters of the RNN:", sum(p.numel() for p in rnn.parameters()))
show_memory(beats[0], out[0])

🔧 **Change it:**
1. `HIDDEN = 32` - which shapes change? How many parameters now?
2. `STEPS = 70` (only the first half of every beat) - do the parameters change?

<details><summary>👀 What you should see</summary>

- input **(8, 140, 1)**, out **(8, 140, 16)** = 16 memory numbers at every one of the 140 steps, scores **(8, 5)**
- **304** parameters = 16·1 (input weights) + 16·16 (memory weights) + 2·16 (biases)
- the picture: the memory lines move most at the sharp peaks of the ECG - between the peaks they settle down.
  The RNN is not trained yet (random weights) - training decides *what* the memory keeps
- HIDDEN = 32: out (8, 140, 32), **1 120** parameters - the memory weights grow with HIDDEN²
- STEPS = 70: out (8, 70, 16), still **304** parameters - the same weights are used at every time step, so the
  RNN works for any length. (A Linear layer on the whole beat would need new weights for every length.)
- `out[:, -1]` = the memory after the last step: it has "read" the whole beat - this is our many-to-one readout
</details>

## 💻 Code break 3 - Vanishing and exploding gradients
We unroll an RNN cell over 50 time steps, compute a loss at the **last** step and measure how big the gradient
is when it arrives at every earlier memory `h_t`. `SIZE` scales the memory weights.

In [ ]:
SIZE = 0.5           # 👈 0.5 (small), 1.0, 3.0 (large)

cell = make_rnn_cell(SIZE)
x = torch.randn(64, 50, 1)                    # 64 random sequences of 50 steps
h = torch.zeros(64, 32)
memories = []
for t in range(50):
    h = cell(x[:, t], h)                      # one step of the RNN
    h.retain_grad()                           # keep the gradient of every h_t
    memories.append(h)
h.sum().backward()                            # a loss at the last step
sizes = [m.grad.norm(dim=1).mean().item() for m in memories]
print(f"gradient at t = 50: {sizes[-1]:.2g}    at t = 1: {sizes[0]:.2g}")
plot_gradients(sizes)

🔧 **Change it:** `SIZE = 1.0`, then `SIZE = 3.0`. Compare the gradient at t = 1.

<details><summary>👀 What you should see</summary>

- SIZE 0.5: gradient **5.7** at t = 50, about **3e-15** at t = 1 - a straight line down on the log axis.
  The gradient is multiplied by the small weights at every step: **vanishing gradient**. The first time steps
  get no learning signal - the RNN cannot learn to remember them.
- SIZE 1.0: about **0.09** at t = 1 - it still shrinks, but slowly
- SIZE 3.0: about **1e7** at t = 1 - a straight line *up*: **exploding gradient**. One training step with such a
  gradient throws the weights far away.
- there is no good fixed size: this is why plain RNNs have a short memory - and why LSTM and GRU exist (block 2)
</details>

## 💻 Code break 4 - Gradient clipping: one line against exploding gradients
`clip_grad_norm_` shrinks the whole gradient if it is longer than `MAX_NORM` - the direction stays the same.
It goes between `loss.backward()` and `opt.step()` (here: after the `backward()` of code break 3).

In [ ]:
MAX_NORM = 1.0       # 👈

grad_before = cell.weight_hh.grad[0, :16].clone()                     # 16 numbers of the gradient, before clipping
before = nn.utils.clip_grad_norm_(cell.parameters(), MAX_NORM)       # returns the length BEFORE clipping
after = torch.cat([p.grad.flatten() for p in cell.parameters()]).norm()
print(f"gradient length before: {before:.3g}    after clipping: {after:.3g}")
show_clipping(grad_before, cell.weight_hh.grad[0, :16])

🔧 **Change it:**
1. Run code break 3 with `SIZE = 3.0`, then this cell again.
2. `MAX_NORM = 100` (run code break 3 again first - clipping changes the gradient) - what changes?

<details><summary>👀 What you should see</summary>

- SIZE 0.5: before about **630**, after **1** (the loss in code break 3 is a sum over 64 × 32 numbers - so it is big)
- the picture: the 16 bars have **exactly the same pattern** before and after - only the scale of the y-axis
  changes (largest bar 5.13 → 0.0082, that is 628 × smaller). Same direction, shorter step.
- SIZE 3.0: before about **6e8** (!), after **1** - every training step is now at most "1 long"
- MAX_NORM = 100: after = **100**. Clipping only shortens gradients that are *longer* than MAX_NORM - normal-sized
  gradients are not touched
- clipping fixes **exploding** gradients only. Vanishing gradients need a better cell: LSTM or GRU
- in the training loop: `loss.backward()` → `nn.utils.clip_grad_norm_(model.parameters(), 1.0)` → `opt.step()`
</details>